# OpenAI extraction workflow

This template builds a small corpus and extracts structured records with OpenAI text and vision profiles. Set `OPENAI_API_KEY` in the environment before starting Jupyter; never paste it into this notebook. Replace the model placeholder with a model available to your account.

In [ ]:
%env PMT_DB=papers.db
%env PMT_QUERY=lithium solid electrolyte
%env PMT_RECIPE=sse
%env PMT_MODEL=YOUR_OPENAI_MODEL
%env PMT_OUTPUT=temp_openai_materials.csv
%env PMT_FINAL=openai_materials.csv

## Configure model profiles

The same model may be used for both profiles only if it accepts images. Otherwise set separate text and vision identifiers.

In [ ]:
%%bash
set -euo pipefail
test "$PMT_MODEL" != "YOUR_OPENAI_MODEL"
pmt config model text --provider openai --model "$PMT_MODEL"
pmt config model vision --provider openai --model "$PMT_MODEL"
pmt config status

## Build and inspect the corpus

OpenAlex can search without a key. Configure other providers separately if you want broader coverage. Start with a small count, inspect the corpus, and scale only after the complete workflow succeeds.

In [ ]:
%%bash
set -euo pipefail
pmt search "$PMT_QUERY" "$PMT_DB" --source openalex --count 25
pmt download "$PMT_DB" --format both
pmt corpus stats "$PMT_DB"

## Scrape and store

`text-images` uses downloaded text and PDF-derived images. Change the mode to `text` for a cheaper first run. The same recipe is passed to storage so aliases and unit conversions remain consistent.

In [ ]:
%%bash
set -euo pipefail
pmt scrape "$PMT_DB" "$PMT_RECIPE" --mode text-images --image-context paper-text --count 5 --output "$PMT_OUTPUT"
pmt store "$PMT_DB" "$PMT_OUTPUT" "$PMT_FINAL" "$PMT_RECIPE" --assume-yes
pmt status "$PMT_DB"